OpenAI API 구현

In [ ]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import requests

load_dotenv()
# 사용할 키 가져오기
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# 연결된 OpenAI 객체 생성
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
import os
import requests
from dotenv import load_dotenv

# .env 파일에서 API 키 불러오기
load_dotenv()

def get_today_weather():
    """
    OpenWeatherMap API를 사용하여 대전의 현재 날씨를
    Streamlit 사이드바용 딕셔너리로 받아오는 함수
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")
    
    # 대전오월드의 고정 위도(lat)와 경도(lon)
    lat = 36.2875
    lon = 127.3985
    
    # units=metric을 반드시 넣어야 섭씨(℃) 온도로 나옵니다! (안 넣으면 켈빈(K)으로 나옴)
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={api_key}&units=metric"

    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        
        # API 키 오류 등 정상 응답이 아닐 경우 방어
        if response.status_code != 200:
            print(f"⚠️ OpenWeatherMap API 오류: {data.get('message')}")
            return {"temp": 20.0, "precipitation": 0.0, "humidity": 50} # 기본값 리턴

        # 1. 기온 및 습도 추출
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        
        # 2. 강수량 추출
        # 비가 오지 않는 맑은 날에는 데이터 안에 'rain'이라는 항목 자체가 아예 없습니다.
        # 따라서 data["rain"]["1h"] 라고 쓰면 에러가 나므로, 아래처럼 안전하게(get) 꺼내야 합니다.
        precipitation = data.get("rain", {}).get("1h", 0.0)

        # Streamlit 사이드바용 딕셔너리 생성
        weather_dict = {
            "temp": temp,
            "precipitation": precipitation,
            "humidity": humidity
        }
        return weather_dict

    except Exception as e:
        print(f"⚠️ 날씨 데이터를 불러오는 중 통신 오류 발생: {e}")
        # 예외 발생 시 Streamlit 화면이 깨지지 않도록 기본값 반환
        return {
            "temp": 22.0,
            "precipitation": 0.0,
            "humidity": 50
        }

# 테스트 실행
if __name__ == "__main__":
    weather_data = get_today_weather()
    print("📊 Streamlit으로 전달될 딕셔너리 데이터:")
    print(weather_data)

📊 Streamlit으로 전달될 딕셔너리 데이터:
{'temp': 29.2, 'precipitation': 0.0, 'humidity': 50}


In [ ]:
import requests

def get_forecast_weather(target_date):
    """
    대전 오월드 위치를 기준으로 특정 날짜의 낮 12시 날씨를 반환합니다.
    (최대 14일 이내 조회 가능)
    """
    # 대전 오월드 위도/경도
    lat = 36.2875
    lon = 127.3985
    
    # 💡 조회 일자 14일 (forecast_days=14)
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&"
        f"hourly=temperature_2m,relative_humidity_2m,precipitation_probability,weather_code&"
        f"timezone=Asia%2FSeoul&forecast_days=14"
    )

    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        
        # 날씨 코드(WMO)에 맞춰 [한글 설명, 이모지 아이콘]을 매핑하는 함수
        def get_weather_info(code):
            if code == 0:
                return "맑음", "☀️"
            elif code in [1, 2]:
                return "구름 조금", "🌤️"
            elif code == 3:
                return "흐림", "☁️"
            elif code in [45, 48]:
                return "안개", "🌫️"
            elif code in [51, 53, 55, 61, 63, 65, 80, 81, 82]:
                return "비", "🌧️"
            elif code in [71, 73, 75, 85, 86]:
                return "눈", "❄️"
            elif code in [95, 96, 99]:
                return "뇌우", "🌩️"
            return "알 수 없음", "🌡️"

        times = data["hourly"]["time"]
        target_time_str = f"{target_date}T12:00"
        
        if target_time_str in times:
            idx = times.index(target_time_str)
            
            # 한글 설명과 아이콘 추출
            weather_desc, weather_icon = get_weather_info(data["hourly"]["weather_code"][idx])
            
            return {
                "weather": weather_desc,
                "icon": weather_icon,
                "temp": data["hourly"]["temperature_2m"][idx],
                "humidity": data["hourly"]["relative_humidity_2m"][idx],
                "pop": data["hourly"]["precipitation_probability"][idx]
            }
        else:
            return {"weather": "데이터 없음", "icon": "❓", "temp": "-", "humidity": "-", "pop": "-"}

    except Exception as e:
        print(f"오류: {e}")
        return None

# 테스트 실행
if __name__ == "__main__":
    selected_date = "2026-08-25" 
    weather_data = get_forecast_weather(selected_date)
    
    print(f"['{selected_date}'의 날씨]")
    print(weather_data

['2026-08-25'의 날씨]
{'weather': '맑음', 'icon': '☀️', 'temp': 28.1, 'humidity': 66, 'pop': 31}
